<a href="https://colab.research.google.com/github/giannisster/knn-and-centroid-tests-over-cifar10/blob/main/DL_Task_1_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pickle
from sklearn.decomposition import PCA
import joblib

In [ ]:
def load_batch(file_path):
    with open(file_path, 'rb') as f:
        data_dict = pickle.load(f, encoding='bytes')
    # data is a 10000x3072 numpy array of uint8.
    # Each row is a 1D representation of a 32x32 color image.
    X = data_dict[b'data']
    y = np.array(data_dict[b'labels'])
    return X, y

# List of training batch files
train_files = [
    '/content/data_batch_1',
  #  '/content/data_batch_2',
  #    '/content/data_batch_3',
  #    '/content/data_batch_4',
  #    '/content/data_batch_5'
]

# Load and concatenate all training batches
X_train_list, y_train_list = [], []
for file in train_files:
    X, y = load_batch(file)
    X_train_list.append(X)
    y_train_list.append(y)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

# Load test batch
X_test, y_test = load_batch('/content/test_batch')

# normalize data
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0



print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import numpy as np

class myMLP:
    def __init__(self, hidden_layers, input_size, hidden_sizes, output_size, learning_rate):

        self.hidden_layers = hidden_layers
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.output_size = output_size
        self.learning_rate = learning_rate

        self.weights = []
        self.biases = []

        # Initialize weights and biases for each layer

        sizes = [input_size] + hidden_sizes + [output_size]
        for i in range(1, len(sizes)):
            w = np.random.randn(sizes[i-1], sizes[i]) * 0.01
            b = np.zeros((1, sizes[i]))

            self.weights.append(w)
            self.biases.append(b)


    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def softmax(self, x):
        x = x - np.max(x, axis=1, keepdims=True)
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def softmax_derivative(self, x):
        s = self.softmax(x)
        return s * (1 - s)

    def sigmoid_derivative(self, z):
        a = self.sigmoid(z)
        return a * (1 - a)

    def forward(self, x):
        a = x
        self.activations = [x]
        self.z = []
        for i in range(self.hidden_layers + 1):
            self.z.append(np.dot(self.activations[i], self.weights[i]) + self.biases[i])

            if i < self.hidden_layers:
                self.activations.append(self.sigmoid(self.z[i]))
            else:
                self.activations.append(self.softmax(self.z[i]))

        return self.activations[-1]

    def loss(self, y_pred, y_true):
        y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
        return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    def cost_derivative(self, y_pred, y_true):
        return y_pred - y_true

    def backward(self, x, y_true):
        self.forward(x)

        self.deltas = [None] * len(self.weights)

        self.deltas[-1] = self.activations[-1] - y_true

        for i in range(len(self.weights) - 2, -1, -1):
            self.deltas[i] = np.dot(self.deltas[i+1], self.weights[i+1].T) * self.sigmoid_derivative(self.z[i])


    def gradient_descent(self, x):
        m = x.shape[0]

        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * np.dot(self.activations[i].T, self.deltas[i]) / m
            self.biases[i] -= self.learning_rate * np.sum(self.deltas[i], axis=0, keepdims=True) / m

    def fit(self, x, y, epochs, batch_size):
        num_classes = self.output_size
        m = x.shape[0]

        # Μετατροπή των labels σε one-hot encoding
        y_one_hot = np.zeros((m, num_classes), dtype=np.float32)
        y_one_hot[np.arange(m), y] = 1

        for epoch in range(epochs):
            # Ανακάτεμα των δεδομένων σε κάθε epoch
            indices = np.arange(m)
            np.random.shuffle(indices)
            x_shuffled = x[indices]
            y_shuffled = y_one_hot[indices]

            for i in range(0, m, batch_size):
                x_batch = x_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]

                self.backward(x_batch, y_batch)
                self.gradient_descent(x_batch)

            # Υπολογισμός του loss για όλο το dataset στο τέλος του epoch
            predictions = self.forward(x)
            current_loss = self.loss(predictions, y_one_hot)
            print(f"Epoch {epoch+1}/{epochs} - Loss: {current_loss:.4f}")

    def predict(self, x):
        a3 = self.forward(x)
        return np.argmax(a3, axis=1)

    def save_weights(self, filename):
        params = {}

        for i in range(len(self.weights)):
            params[f"W{i}"] = self.weights[i]
            params[f"b{i}"] = self.biases[i]

        np.savez(filename, **params)


    def load_weights(self, filename):
        data = np.load(filename)

        for i in range(len(self.weights)):
            self.weights[i] = data[f"W{i}"]
            self.biases[i] = data[f"b{i}"]

        data.close()

In [ ]:
pca = PCA(n_components=100, svd_solver='randomized', random_state=42)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# joblib.dump(pca, "pca_model.pkl")

print("X_train_pca:", X_train_pca.shape)
print("X_test_pca:", X_test_pca.shape)

print("Explained variance:", np.sum(pca.explained_variance_ratio_))

In [ ]:
mlp = myMLP(100, 128, 64, 10, 0.001)

mlp.fit(X_train_pca, y_train, epochs = 100, batch_size = 50)
mlp.save_weights("my_mlp_weights.npz")

In [ ]:
y_pred = mlp.predict(X_test_pca)
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")